In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys

utils_path = '/content/drive/MyDrive/Colab_Utils'
if utils_path not in sys.path:
    sys.path.insert(0, utils_path)
    print(f"Added {utils_path} to sys.path")
else:
    print(f"{utils_path} already in sys.path")


In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import manual_nms
print("Módulo utils recarregado com a nova implementação de manual_nms.")

In [ ]:
import os
import random
import colorsys
import numpy as np
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from utils import manual_nms

# Definição de caminhos base para evitar erros de arquivo não encontrado
BASE_PATH = '/content/drive/MyDrive/YOLO_MS671/'

# ==========================================
# 1. ÂNCORAS DO YOLOV3 (COCO)
# ==========================================
YOLOV3_ANCHORS = [
    [(116, 90), (156, 198), (373, 326)],
    [(30, 61), (62, 45), (59, 119)],
    [(10, 13), (16, 30), (33, 23)]
]

# ==========================================
# 2. FUNÇÕES DE UTILIDADE E PRÉ-PROCESSAMENTO
# ==========================================
def read_classes(classes_path):
    with open(classes_path) as f:
        return [c.strip() for c in f.readlines()]

def generate_colors(class_names):
    hsv_tuples = [(x / len(class_names), 1., 1.) for x in range(len(class_names))]
    colors = list(map(lambda x: colorsys.hsv_to_rgb(*x), hsv_tuples))
    colors = list(map(lambda x: (int(x[0] * 255), int(x[1] * 255), int(x[2] * 255)), colors))
    random.seed(10101)
    random.shuffle(colors)
    random.seed(None)
    return colors

def letterbox_image(image, size=(416, 416)):
    iw, ih = image.size
    w, h = size
    scale = min(w / iw, h / ih)
    nw, nh = int(iw * scale), int(ih * scale)
    image = image.resize((nw, nh), Image.BICUBIC)
    new_image = Image.new('RGB', size, (128, 128, 128))
    new_image.paste(image, ((w - nw) // 2, (h - nh) // 2))
    return new_image

def reverter_escala_caixas(boxes, img_size, original_shape):
    iw, ih = original_shape
    w, h = img_size
    scale = min(w / iw, h / ih)
    nw, nh = int(iw * scale), int(ih * scale)
    dx, dy = (w - nw) / 2.0 / w, (h - nh) / 2.0 / h
    scale_w, scale_h = nw / w, nh / h
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - dy) / scale_h
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] - dx) / scale_w
    boxes[:, [0, 2]] *= ih
    boxes[:, [1, 3]] *= iw
    return boxes

def preprocess_image(img_path, model_image_size=(416, 416)):
    image = Image.open(img_path).convert('RGB')
    boxed_image = letterbox_image(image, model_image_size)
    image_data = np.array(boxed_image, dtype='float32') / 255.0
    image_data = image_data[:, :, ::-1].copy()
    image_data = np.transpose(image_data, (2, 0, 1))
    image_data = torch.from_numpy(image_data).unsqueeze(0)
    return image, image_data

# ==========================================
# 3. BLOCOS DE CONSTRUÇÃO DA REDE
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p, bn=True):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, bias=not bn)
        self.bn = nn.BatchNorm2d(out_c, eps=1e-5) if bn else None
        self.act = nn.LeakyReLU(0.1, inplace=True) if bn else None
    def forward(self, x):
        x = self.conv(x)
        if self.bn: x = self.act(self.bn(x))
        return x

class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBlock(channels, channels // 2, 1, 1, 0)
        self.conv2 = ConvBlock(channels // 2, channels, 3, 1, 1)
    def forward(self, x): return x + self.conv2(self.conv1(x))

class YOLOv3(nn.Module):
    def __init__(self, num_classes=80):
        super().__init__()
        self.conv1 = ConvBlock(3, 32, 3, 1, 1)
        self.layer1 = self._make_layer(32, 64, 1)
        self.layer2 = self._make_layer(64, 128, 2)
        self.layer3 = self._make_layer(128, 256, 8)
        self.layer4 = self._make_layer(256, 512, 8)
        self.layer5 = self._make_layer(512, 1024, 4)
        self.head1_1 = self._make_c5(1024, 512)
        self.head1_2 = self._make_yolo_head(512, 1024, num_classes)
        self.head2_1 = ConvBlock(512, 256, 1, 1, 0)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.head2_2 = self._make_c5(768, 256)
        self.head2_3 = self._make_yolo_head(256, 512, num_classes)
        self.head3_1 = ConvBlock(256, 128, 1, 1, 0)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.head3_2 = self._make_c5(384, 128)
        self.head3_3 = self._make_yolo_head(128, 256, num_classes)

    def _make_layer(self, in_c, out_c, num_blocks):
        layers = [ConvBlock(in_c, out_c, 3, 2, 1)]
        for _ in range(num_blocks): layers.append(ResBlock(out_c))
        return nn.Sequential(*layers)

    def _make_c5(self, in_c, out_c):
        return nn.Sequential(ConvBlock(in_c, out_c, 1, 1, 0), ConvBlock(out_c, out_c * 2, 3, 1, 1), ConvBlock(out_c * 2, out_c, 1, 1, 0), ConvBlock(out_c, out_c * 2, 3, 1, 1), ConvBlock(out_c * 2, out_c, 1, 1, 0))

    def _make_yolo_head(self, in_c, out_c, num_classes):
        return nn.Sequential(ConvBlock(in_c, out_c, 3, 1, 1), nn.Conv2d(out_c, 3 * (5 + num_classes), 1, 1, 0, bias=True))

    def forward(self, x):
        x = self.layer2(self.layer1(self.conv1(x)))
        r1, r2 = self.layer3(x), self.layer4(self.layer3(x))
        x = self.layer5(r2)
        x1_5 = self.head1_1(x)
        out1 = self.head1_2(x1_5)
        x = self.upsample1(self.head2_1(x1_5))
        x2_5 = self.head2_2(torch.cat([x, r2], dim=1))
        out2 = self.head2_3(x2_5)
        x = self.upsample2(self.head3_1(x2_5))
        x3_5 = self.head3_2(torch.cat([x, r1], dim=1))
        out3 = self.head3_3(x3_5)
        return out1, out2, out3

def carregar_pesos_yolov3(caminho_weights, modelo):
    with open(caminho_weights, "rb") as f:
        header = np.fromfile(f, dtype=np.int32, count=5)
        weights = np.fromfile(f, dtype=np.float32)
    modulos = [m for m in modelo.modules() if isinstance(m, ConvBlock) or (isinstance(m, nn.Conv2d) and m.bias is not None)]
    ptr = 0
    for modulo in modulos:
        if isinstance(modulo, ConvBlock):
            conv, bn = modulo.conv, modulo.bn
            for param in [bn.bias, bn.weight, bn.running_mean, bn.running_var]:
                num = param.numel()
                param.data.copy_(torch.from_numpy(weights[ptr:ptr+num]).view_as(param))
                ptr += num
            num = conv.weight.numel()
            conv.weight.data.copy_(torch.from_numpy(weights[ptr:ptr+num]).view_as(conv.weight))
            ptr += num
        elif isinstance(modulo, nn.Conv2d):
            for param in [modulo.bias, modulo.weight]:
                num = param.numel()
                param.data.copy_(torch.from_numpy(weights[ptr:ptr+num]).view_as(param))
                ptr += num
    return modelo

def decode_yolo(feats, anchors, num_classes, img_size=416):
    B, C, H, W = feats.shape
    num_anchors = len(anchors)
    feats = feats.view(B, num_anchors, 5 + num_classes, H, W).permute(0, 1, 3, 4, 2).contiguous()
    grid_y, grid_x = torch.meshgrid(torch.arange(H), torch.arange(W), indexing='ij')
    grid = torch.stack((grid_x, grid_y), dim=-1).float().to(feats.device).view(1, 1, H, W, 2)
    box_xy = (torch.sigmoid(feats[..., :2]) + grid) / torch.tensor([W, H], device=feats.device)
    anchors_tensor = torch.tensor(anchors, device=feats.device).view(1, num_anchors, 1, 1, 2)
    box_wh = (torch.exp(torch.clamp(feats[..., 2:4], max=15.0)) * anchors_tensor) / img_size
    scores = torch.sigmoid(feats[..., 4:5]) * torch.sigmoid(feats[..., 5:])
    box_mins, box_maxes = box_xy - (box_wh / 2.), box_xy + (box_wh / 2.)
    boxes = torch.cat([box_mins[..., 1:2], box_mins[..., 0:1], box_maxes[..., 1:2], box_maxes[..., 0:1]], dim=-1)
    return boxes.view(-1, 4), scores.view(-1, num_classes)

def executar_predicao(image_file, model, class_names, device, score_threshold=0.5, iou_threshold=0.4, output_path=None):
    # Garante o uso do caminho base se o arquivo não for absoluto
    if not os.path.isabs(image_file):
        image_file = os.path.join(BASE_PATH, image_file)

    image, image_data = preprocess_image(image_file, (416, 416))
    model.eval()
    with torch.no_grad():
        o1, o2, o3 = model(image_data.to(device))
        b1, s1 = decode_yolo(o1, YOLOV3_ANCHORS[0], len(class_names))
        b2, s2 = decode_yolo(o2, YOLOV3_ANCHORS[1], len(class_names))
        b3, s3 = decode_yolo(o3, YOLOV3_ANCHORS[2], len(class_names))
        boxes, scores = torch.cat([b1, b2, b3]), torch.cat([s1, s2, s3])
        mask = (scores.max(-1)[0] >= score_threshold) & torch.isfinite(boxes).all(-1)
        boxes, scores = boxes[mask], scores[mask]
        if boxes.size(0) == 0: return print("Nada detectado.")
        boxes = reverter_escala_caixas(boxes, (416, 416), image.size)
        s_vals, c_ids = scores.max(-1)
        keep = manual_nms(boxes, s_vals, iou_threshold).long()[:10]
        boxes, scores, classes = boxes[keep], s_vals[keep], c_ids[keep]

    draw = ImageDraw.Draw(image)
    font_size = max(20, int(min(image.size) * 0.04))
    try:
        font = ImageFont.truetype("LiberationSans-Regular.ttf", font_size)
    except:
        font = ImageFont.load_default()

    thickness = max(3, int(min(image.size) * 0.005))
    colors = generate_colors(class_names)

    for i, c in enumerate(classes.cpu().numpy()):
        box, score = boxes[i].cpu().numpy(), scores[i].cpu().item()
        label = f'{class_names[c]} {score:.2f}'
        top, left, bottom, right = [int(v) for v in box]
        for j in range(thickness): draw.rectangle([left+j, top+j, right-j, bottom-j], outline=colors[c])
        bbox = draw.textbbox((left, top), label, font=font)
        draw.rectangle([bbox[0], bbox[1]-2, bbox[2]+2, bbox[3]+2], fill=colors[c])
        draw.text((left, top - (bbox[3]-bbox[1])), label, fill=(0,0,0), font=font)

    if output_path:
        image.save(os.path.join(BASE_PATH, output_path))
        print(f"Imagem salva em: {output_path}")

    plt.figure(figsize=(12, 12)); plt.imshow(image); plt.axis('off'); plt.show()

# ==========================================
# 6. EXECUÇÃO PRINCIPAL
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = read_classes(os.path.join(BASE_PATH, "coco.names"))
modelo = YOLOv3(num_classes=len(class_names)).to(device)

try:
    weights_file = os.path.join(BASE_PATH, "yolov3.weights")
    pth_file = os.path.join(BASE_PATH, "yolov3_convertido.pth")
    if not os.path.exists(pth_file):
        modelo = carregar_pesos_yolov3(weights_file, modelo)
        torch.save(modelo.state_dict(), pth_file)
    modelo.load_state_dict(torch.load(pth_file, map_location=device))
except Exception as e:
    print(f"Erro no carregamento: {e}")

# Teste
executar_predicao("betao_cadeira.jpg", modelo, class_names, device, score_threshold=0.5)

### Painel de Testes Interativo
Ajuste os parâmetros abaixo para observar como a detecção se comporta.
- **Confiança (Score)**: Filtra detecções fracas.
- **IoU (NMS)**: Define o limite de sobreposição para considerar duas caixas como o mesmo objeto.

In [ ]:
#@title Configurações de Inferência { run: "auto" }

nome_da_imagem = "city_scene.jpg" #@param {type:"string"}
confianca = 0.66 #@param {type:"slider", min:0.01, max:1, step:0.05}
limiar_nms = 0.51 #@param {type:"slider", min:0.01, max:1, step:0.05}
salvar_resultado = True #@param {type:"boolean"}

output_file = "city_result.jpg" if salvar_resultado else None

executar_predicao(
    nome_da_imagem,
    modelo,
    class_names,
    device,
    score_threshold=confianca,
    iou_threshold=limiar_nms,
    output_path=output_file
)

In [ ]:
# Re-executando a predição após a correção do NMS
executar_predicao("betao_cadeira.jpg", modelo, class_names, device, score_threshold=0.5)

In [ ]:
# Listando todas as classes que o modelo foi treinado para reconhecer
print(f'O modelo foi treinado para reconhecer {len(class_names)} classes:')
for i, name in enumerate(class_names):
    print(f'{i+1}: {name}')

In [ ]:
# Código para depurar o tipo de dado retornado pelo manual_nms
# Certifique-se de que a célula com a definição da função manual_nms (em utils.py) já foi executada.
import torch
from utils import manual_nms
try:
    # Vamos criar dados fictícios para testar a função
    test_boxes = torch.tensor([[10, 10, 50, 50], [15, 15, 55, 55]], dtype=torch.float32)
    test_scores = torch.tensor([0.9, 0.8], dtype=torch.float32)

    keep = manual_nms(test_boxes, test_scores, 0.5)

    print(f"Tipo da variável 'keep': {type(keep)}")
    if isinstance(keep, torch.Tensor):
        print(f"Dtype do Tensor 'keep': {keep.dtype}")
        print(f"Conteúdo de 'keep': {keep}")
    elif isinstance(keep, list):
        print(f"Conteúdo da lista 'keep': {keep}")
        if len(keep) > 0:
             print(f"Tipo do primeiro elemento da lista: {type(keep[0])}")
except Exception as e:
    print(f"Erro ao testar manual_nms: {e}")